In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
OLED显示系统信息 - 简单可靠版本
"""

import time
import subprocess
from i2cdev import I2C

# 配置
I2C_BUS = 5
OLED_ADDR = 0x3C

def oled_write_cmd(i2c, cmd):
    i2c.write(bytes([0x00, cmd]))

def oled_write_data(i2c, data):
    i2c.write(bytes([0x40, data]))

def oled_set_pos(i2c, page, col):
    oled_write_cmd(i2c, 0xB0 + page)
    oled_write_cmd(i2c, 0x00 + (col & 0x0F))
    oled_write_cmd(i2c, 0x10 + (col >> 4))

def oled_init(i2c):
    """初始化OLED"""
    cmds = [
        0xAE, 0x20, 0x00, 0xB0, 0xC8, 0x00, 0x10, 0x40,
        0x81, 0xCF, 0xA1, 0xA6, 0xA8, 0x3F, 0xA4, 0xD3,
        0x00, 0xD5, 0xF0, 0xD9, 0x22, 0xDA, 0x12, 0xDB,
        0x20, 0x8D, 0x14, 0xAF
    ]
    for cmd in cmds:
        oled_write_cmd(i2c, cmd)
        time.sleep(0.001)

# 5x7字体（简单可靠）
FONT = {
    '0': [0x3E, 0x51, 0x49, 0x45, 0x3E],
    '1': [0x00, 0x42, 0x7F, 0x40, 0x00],
    '2': [0x42, 0x61, 0x51, 0x49, 0x46],
    '3': [0x21, 0x41, 0x45, 0x4B, 0x31],
    '4': [0x18, 0x14, 0x12, 0x7F, 0x10],
    '5': [0x27, 0x45, 0x45, 0x45, 0x39],
    '6': [0x3C, 0x4A, 0x49, 0x49, 0x30],
    '7': [0x01, 0x71, 0x09, 0x05, 0x03],
    '8': [0x36, 0x49, 0x49, 0x49, 0x36],
    '9': [0x06, 0x49, 0x49, 0x29, 0x1E],
    'A': [0x7E, 0x11, 0x11, 0x11, 0x7E],
    'B': [0x7F, 0x49, 0x49, 0x49, 0x36],
    'C': [0x3E, 0x41, 0x41, 0x41, 0x22],
    'D': [0x7F, 0x41, 0x41, 0x22, 0x1C],
    'E': [0x7F, 0x49, 0x49, 0x49, 0x41],
    'F': [0x7F, 0x09, 0x09, 0x09, 0x01],
    'G': [0x3E, 0x41, 0x49, 0x49, 0x7A],
    'H': [0x7F, 0x08, 0x08, 0x08, 0x7F],
    'I': [0x00, 0x41, 0x7F, 0x41, 0x00],
    'L': [0x7F, 0x40, 0x40, 0x40, 0x40],
    'M': [0x7F, 0x02, 0x0C, 0x02, 0x7F],
    'N': [0x7F, 0x04, 0x08, 0x10, 0x7F],
    'O': [0x3E, 0x41, 0x41, 0x41, 0x3E],
    'P': [0x7F, 0x09, 0x09, 0x09, 0x06],
    'R': [0x7F, 0x09, 0x19, 0x29, 0x46],
    'S': [0x46, 0x49, 0x49, 0x49, 0x31],
    'T': [0x01, 0x01, 0x7F, 0x01, 0x01],
    'U': [0x3F, 0x40, 0x40, 0x40, 0x3F],
    'W': [0x3F, 0x40, 0x38, 0x40, 0x3F],
    'Y': [0x07, 0x08, 0x70, 0x08, 0x07],
    ' ': [0x00, 0x00, 0x00, 0x00, 0x00],
    '.': [0x00, 0x60, 0x60, 0x00, 0x00],
    ':': [0x00, 0x36, 0x36, 0x00, 0x00],
    '/': [0x20, 0x10, 0x08, 0x04, 0x02],
    '%': [0x42, 0x25, 0x12, 0x48, 0x21],
    '-': [0x08, 0x08, 0x08, 0x08, 0x08],
    '>': [0x00, 0x41, 0x22, 0x14, 0x08],
}

def draw_text(i2c, text, page, col=0):
    """在指定位置绘制文本"""
    for char in text:
        if col >= 128:
            break
        if char in FONT:
            oled_set_pos(i2c, page, col)
            for i in range(5):
                oled_write_data(i2c, FONT[char][i])
            oled_write_data(i2c, 0x00)  # 间距
        col += 6

def draw_big_text(i2c, text, page, col=0):
    """绘制大号文本（2倍高度）"""
    for char in text:
        if col >= 128:
            break
        if char in FONT:
            # 上半部分
            oled_set_pos(i2c, page, col)
            for i in range(5):
                oled_write_data(i2c, FONT[char][i])
            oled_write_data(i2c, 0x00)
            
            # 下半部分
            oled_set_pos(i2c, page + 1, col)
            for i in range(5):
                oled_write_data(i2c, FONT[char][i])
            oled_write_data(i2c, 0x00)
        col += 6

def oled_clear(i2c):
    """快速清屏"""
    for page in range(8):
        oled_set_pos(i2c, page, 0)
        for col in range(128):
            oled_write_data(i2c, 0x00)

def get_cpu_temp():
    """获取CPU温度"""
    try:
        with open('/sys/class/thermal/thermal_zone0/temp', 'r') as f:
            return float(f.read()) / 1000
    except:
        return 0.0

def get_memory_info():
    """获取内存信息"""
    try:
        result = subprocess.run(['free', '-m'], capture_output=True, text=True)
        lines = result.stdout.split('\n')
        if len(lines) > 1:
            parts = lines[1].split()
            if len(parts) >= 3:
                total = int(parts[1])
                used = int(parts[2])
                percent = (used / total * 100) if total > 0 else 0
                return total, used, percent
    except:
        pass
    return 0, 0, 0

def get_cpu_usage():
    """获取CPU使用率"""
    try:
        with open('/proc/stat', 'r') as f:
            line = f.readline()
            parts = line.split()[1:5]
            total = sum(int(p) for p in parts)
            idle = int(parts[3])
            return (1 - idle/total) * 100 if total > 0 else 0
    except:
        return 0.0

def main():
    print("OLED System Monitor")
    print(f"Bus: {I2C_BUS}, Addr: 0x{OLED_ADDR:02X}")
    
    try:
        i2c = I2C(OLED_ADDR, I2C_BUS)
        oled_init(i2c)
        
        print("OLED initialized. Press Ctrl+C to exit.")
        
        last_temp = -1
        last_mem = -1
        
        while True:
            # 获取数据
            cpu_temp = get_cpu_temp()
            mem_total, mem_used, mem_percent = get_memory_info()
            
            # 只在数据变化时更新
            if abs(cpu_temp - last_temp) > 0.5 or abs(mem_percent - last_mem) > 1:
                oled_clear(i2c)
                
                # 标题
                draw_text(i2c, "SYSTEM MONITOR", 0, 0)
                
                # CPU温度 - 大字体显示
                draw_text(i2c, "CPU TEMP:", 2, 0)
                draw_big_text(i2c, f"{cpu_temp:.1f}C", 4, 0)
                
                # 内存信息
                draw_text(i2c, "MEMORY:", 6, 0)
                mem_text = f"{mem_used}/{mem_total}M ({mem_percent:.0f}%)"
                draw_text(i2c, mem_text, 7, 0)
                
                last_temp = cpu_temp
                last_mem = mem_percent
            
            time.sleep(0.5)  # 检测间隔
            
    except KeyboardInterrupt:
        print("\nStopped")
        oled_clear(i2c)
        i2c.close()
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()

OLED System Monitor
Bus: 5, Addr: 0x3C
OLED initialized. Press Ctrl+C to exit.

Stopped
